In [ ]:
# https://www.matecdev.com/posts/landsat-sentinel-aws-s3-python.html
from pystac_client import Client
from json import load
import requests
from pyproj import Transformer
import rasterio as rio

In [2]:
LandsatSTAC = Client.open("https://landsatlook.usgs.gov/stac-server", headers=[])

for collection in LandsatSTAC.get_collections():
    print(collection)

<CollectionClient id=landsat-c2l2-sr>
<CollectionClient id=landsat-c2l2-st>
<CollectionClient id=landsat-c2ard-st>
<CollectionClient id=landsat-c2l2alb-bt>
<CollectionClient id=landsat-c2l3-fsca>
<CollectionClient id=landsat-c2ard-bt>
<CollectionClient id=landsat-c2l1>
<CollectionClient id=landsat-c2l3-ba>
<CollectionClient id=landsat-c2l2alb-st>
<CollectionClient id=landsat-c2ard-sr>
<CollectionClient id=landsat-c2l2alb-sr>
<CollectionClient id=landsat-c2l2alb-ta>
<CollectionClient id=landsat-c2l3-dswe>
<CollectionClient id=landsat-c2ard-ta>


In [3]:
def BuildSquare(lon, lat, delta):
    c1 = [lon + delta, lat + delta]
    c2 = [lon + delta, lat - delta]
    c3 = [lon - delta, lat - delta]
    c4 = [lon - delta, lat + delta]
    geometry = {"type": "Polygon", "coordinates": [[ c1, c2, c3, c4, c1 ]]}
    return geometry

geometry = BuildSquare(-59.346271, -34.233076, 0.04)
timeRange = '2019-06-01/2021-06-01'

In [4]:
LandsatSearch = LandsatSTAC.search ( 
    intersects = geometry,
    datetime = timeRange,
    query =  ['eo:cloud_cover95'],
    collections = ["landsat-c2l2-sr"] )

Landsat_items = [i.to_dict() for i in LandsatSearch.items()]
print(f"{len(Landsat_items)} Landsat scenes fetched")

193 Landsat scenes fetched


In [12]:
for item in Landsat_items:
    red_href = item['assets']['red']['href']
    red_s3 = item['assets']['red']['alternate']['s3']['href']
    print(red_href)    
    print(red_s3)

https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF
s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF
https://landsatlook.usgs.gov/data/collection02/level-2/standard/etm/2021/226/084/LE07_L2SP_226084_20210527_20210622_02_T1/LE07_L2SP_226084_20210527_20210622_02_T1_SR_B3.TIF
s3://usgs-landsat/collection02/level-2/standard/etm/2021/226/084/LE07_L2SP_226084_20210527_20210622_02_T1/LE07_L2SP_226084_20210527_20210622_02_T1_SR_B3.TIF
https://landsatlook.usgs.gov/data/collection02/level-2/standard/etm/2021/225/084/LE07_L2SP_225084_20210520_20210615_02_T1/LE07_L2SP_225084_20210520_20210615_02_T1_SR_B3.TIF
s3://usgs-landsat/collection02/level-2/standard/etm/2021/225/084/LE07_L2SP_225084_20210520_20210615_02_T1/LE07_L2SP_225084_20210520_20210615_02_T1_SR_B3.TIF


In [13]:
def download_landsat(landsat_url, download_path):
    response = requests.get(landsat_url, stream=True)
    if response.status_code == 200:
        with open(download_path, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
    else:
        raise Exception(f"Failed to download Landsat data. Status code: {response.status_code}")

In [ ]:

from pyproj import Transformer

def getSubset(geotiff_file, bbox):
    with rio.open(geotiff_file) as geo_fp:
        # Calculate pixels with PyProj
        Transf = Transformer.from_crs("epsg:4326", geo_fp.crs)
        lat_north, lon_west = Transf.transform(bbox[3], bbox[0])
        lat_south, lon_east = Transf.transform(bbox[1], bbox[2])
        x_top, y_top = geo_fp.index(lat_north, lon_west)
        x_bottom, y_bottom = geo_fp.index(lat_south, lon_east)
        
        # Define window in RasterIO
        window = rio.windows.Window.from_slices((x_top, x_bottom), (y_top, y_bottom))
        
        # Read the subset
        subset = geo_fp.read(1, window=window)
    
    return subset


ModuleNotFoundError: No module named 'rasterio'